# OptionResets Pitch EDA: do option-overlay resets move the S&P close?

**Mechanisms.**
* **Buffer / defined-outcome ETFs** (about $60–70bn) reset monthly series at the close of the first trading day of
  the month, buying new FLEX option packages; dealers hedge the delta at the close.
* **JPMorgan Hedged Equity** rolls its large SPX put-spread collar at the close of the last business day of each quarter.

**Design.** SPY 5-minute bars 2020–2026 (`IndexGamma Pitch/data`), official closes from Yahoo. Last 30 / 60 minutes and
absolute last-30 move on reset days vs ordinary days; other month-ends as a comparison.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from scipy import stats
x=pd.read_csv("../IndexGamma Pitch/data/intraday/SPY.csv",parse_dates=["datetime"]); x["d"]=x.datetime.dt.normalize(); x["hm"]=x.datetime.dt.strftime("%H:%M")
c=x.pivot_table(index="d",columns="hm",values="close"); c=c[c.notna().sum(1)>=76]
cl=pd.read_csv("../TOM Pitch/data/etfs.csv",parse_dates=["date"]).pivot(index="date",columns="ticker",values="close").SPY
C=cl.reindex(c.index); P=cl.shift(1).reindex(c.index)
D=pd.DataFrame({"last30":np.log(C/c["15:25"])*1e4,"last60":np.log(C/c["14:55"])*1e4,"day":np.log(C/P)*1e4,"abs30":np.abs(np.log(C/c["15:25"])*1e4)})
s=pd.Series(D.index,index=D.index); ym=D.index.to_period("M")
first=s.groupby(ym).transform("min")==s; last=s.groupby(ym).transform("max")==s
qend=last&D.index.month.isin([3,6,9,12])
D["kind"]=np.select([first,qend,last],["first day (buffer resets)","quarter-end (JPM collar)","other month-end"],"other day")
nxt=D.shift(-1)
for k in ["first day (buffer resets)","quarter-end (JPM collar)","other month-end"]:
    a=D[D.kind==k]; b=D[D.kind=="other day"]
    print(f"{k:28s} n {len(a):3d}: " + " | ".join(f"{c} {a[c].mean():+.1f} vs {b[c].mean():+.1f} (t {stats.ttest_ind(a[c].dropna(),b[c].dropna(),equal_var=False)[0]:+.2f})" for c in ["last30","last60","abs30"]))

first day (buffer resets)    n  81: last30 +3.0 vs -0.6 (t +0.93) | last60 +5.2 vs -1.2 (t +1.10) | abs30 +20.6 vs +19.7 (t +0.29)
quarter-end (JPM collar)     n  27: last30 -6.6 vs -0.6 (t -0.86) | last60 -6.9 vs -1.2 (t -0.65) | abs30 +27.1 vs +19.7 (t +1.58)
other month-end              n  54: last30 +10.4 vs -0.6 (t +1.57) | last60 +11.4 vs -1.2 (t +1.85) | abs30 +34.0 vs +19.7 (t +2.64)


**Interpretation.** Neither reset leaves a mark: first-day closes (+3bp, t 0.9) and quarter-end closes (−7bp, t −0.9)
match ordinary days, and quarter-end volatility is only mildly higher (t 1.6). Ordinary month-ends are the ones that
stand out (+11bp in the last hour, t 1.9; volatility up, t 2.6): the generic month-end closing imbalance, not the
option overlays.

**Verdict: dead.**